# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to explore and analyze the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, referencing data entities with their `@id` fields as recommended by the Croissant specification.

### Dataset Source
The dataset's Croissant schema is provided at:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` and pandas are installed
!pip install mlcroissant pandas --quiet

## 1. Data Loading
Load the dataset metadata and records from the provided Croissant schema URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Dataset
ds = mlc.Dataset(croissant_url)
metadata = ds.metadata  # Metadata as an object

print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}\nLicense: {metadata.license}")


## 2. Data Overview
Review available **record sets**, **fields**, and their `@id`s for reference and downstream extraction.

In [ ]:
# For the FAIR^2 Croissant schema, enumerate the available record sets and their fields using their @id attributes.

def list_record_sets(dataset):
    record_sets = dataset.metadata.recordSets
    if not record_sets:
        print("No record sets found in the Croissant schema.")
        return []
    for rs in record_sets:
        print(f"RecordSet name: {rs.name}")
        print(f"  @id: {rs['@id']}")
        print(f"  Description: {rs.description if hasattr(rs, 'description') else '-'}")
        print(f"  Fields:")
        for f in rs.fields:
            print(f"    - {f.name} (@id: {f['@id']}, dataType: {getattr(f, 'dataType', '-')})")
        print()
    return [rs['@id'] for rs in record_sets]

record_set_ids = list_record_sets(ds)

# If no record sets present, attempt to load from distribution
if not record_set_ids:
    print("The dataset does not list record sets explicitly.")
    print("Attempting to infer from available distributions...")
    dists = ds.metadata.distribution
    if dists:
        for d in dists:
            # Print out only the available @id's
            print(f"Distribution @id: {d['@id']}")
            print(json.dumps(d, indent=2))

## 3. Data Extraction
Load data from a **specific record set** (referenced by its `@id`) into a dataframe for analysis.

> **Note:** For this [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json), the Croissant schema does not enumerate `recordSet` at the top-level, but distribution files are provided. We reference their `@id` attributes directly for extraction below.

In [ ]:
# Example: Use the distribution's @id as a 'record set' for loading main table data.

# Find all distributions (data tables)
from mlcroissant._src.structure_discovery.croissant_types import Distribution
main_dist = None
main_dist_id = None
for d in ds.metadata.distribution:
    # Pick the main data table by e.g. file type or annotation
    if hasattr(d, 'encodingFormat') and str(d.encodingFormat).startswith('text/tsv'):
        main_dist = d
        main_dist_id = d['@id']
        break
if not main_dist:
    # Just pick the first one
    main_dist = ds.metadata.distribution[0]
    main_dist_id = main_dist['@id']

print(f"Selected distribution for extraction: {main_dist_id}")

# Most Croissant schemas reference recordSet by @id—simulate this with the distribution's @id
record_sets = [main_dist_id]
# For each, extract records
dataframes = {}
for record_set_id in record_sets:
    try:
        records = list(ds.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[record_set_id])} rows for recordSet @id {record_set_id}")
    except Exception as e:
        print(f"Error loading records for record_set '{record_set_id}': {e}")

# Print columns available
if main_dist_id in dataframes:
    print(dataframes[main_dist_id].columns.tolist())
    dataframes[main_dist_id].head()
else:
    print("No dataframe loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps (filtering, normalization, grouping). The fields are referenced by their `@id`.

Assume we want to analyze a numeric field; in this dataset, possible choices might be 'cr:abundance' or 'cr:molecular_weight'.

We will filter for records with high molecular weight (>70), normalize it, and group by a field such as 'cr:accession' or 'cr:sample_id' if available.

In [ ]:
# Identify possible numeric fields from DataFrame
df = dataframes[main_dist_id]
numeric_candidates = [col for col in df.columns if df[col].dtype.kind in 'fi']
print("Numeric field candidates:", numeric_candidates)

# Select a numeric field by @id, e.g. 'cr:molecular_weight' if present
numeric_field = 'cr:molecular_weight' if 'cr:molecular_weight' in df.columns else numeric_candidates[0]
print(f"Using numeric field: {numeric_field}")

threshold = 70
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field, e.g. 'cr:accession' or similar
group_field = 'cr:accession' if 'cr:accession' in df.columns else df.columns[0]
if group_field in df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}: (showing up to 5)")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields. Here, we plot a histogram of the selected numeric field and a scatterplot against another numeric field if present.

In [ ]:
import matplotlib.pyplot as plt

# Histogram
plt.figure(figsize=(7,4))
filtered_df[numeric_field].hist(bins=30)
plt.title(f'Histogram of {numeric_field} (filtered)')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# If available, scatterplot numeric_field vs another numeric column
other_numeric = None
for c in numeric_candidates:
    if c != numeric_field:
        other_numeric = c
        break

if other_numeric:
    plt.figure(figsize=(6,4))
    plt.scatter(filtered_df[numeric_field], filtered_df[other_numeric], alpha=0.6)
    plt.xlabel(numeric_field)
    plt.ylabel(other_numeric)
    plt.title(f'Scatter: {numeric_field} vs {other_numeric}')
    plt.show()
else:
    print('Only one numeric field available; skipping scatter plot.')

## 6. Conclusion

- The FAIR^2 dataset contains mass spectrometry data on extracellular vesicles (EVs) proteins, with rich fields for abundance and protein identification.
- Schema exploration using `mlcroissant` enables programmatic access to record sets, fields, and values via `@id`.
- Basic EDA and visualization uncover ranges and relationships of numeric fields such as molecular weight and abundance.

> For advanced analysis, researchers can further filter, cluster, or model protein features, leveraging the structured Croissant schema for reproducibility and FAIR compliance.